In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pickle

# Configuration

Set your paths and DLC suffix here. The DLC suffix is the part of the CSV filename that comes after the video name.

In [ ]:
# ============ CHANGE THESE PATHS ============
main_dir = '/home/user/Rotarod'

# Directory containing your DLC output CSV files
data_dir = '/Users/gabriela/Desktop/Kirill Lab/Rotarod Data/648LDLC_Resnet50_RotarodAnalysis2Mar2shuffle1_snapshot_best-140'

# Output directories
pickle_dir = main_dir + '/results/pickle/'
plots_dir = main_dir + '/results/plots/'

# The suffix DLC appends to your video name in CSV filenames
# e.g. if CSV is '648LDLC_Resnet50_RotarodAnalysis2Mar2shuffle1_snapshot_best-140.csv'
# and video is '648L.MP4', then the suffix is:
dlc_suffix = 'DLC_Resnet50_RotarodAnalysis2Mar2shuffle1_snapshot_best-140'

# Frame rate of your videos
framerate = 30

# ============================================

Path(pickle_dir).mkdir(parents=True, exist_ok=True)
Path(plots_dir).mkdir(parents=True, exist_ok=True)

# Useful functions

## Function to build a dataframe from a DLC CSV (single animal)

In [ ]:
def build_dataframe(path):
    """Read a DLC CSV file and return times and a tidy DataFrame.
    
    Assumes single-animal tracking. Returns a DataFrame with
    MultiIndex columns: (bodypart, coordinate) where coordinate
    is 'x', 'y', or 'likelihood'.
    """
    with open(path) as f:
        lines = f.readlines()

    # Parse body part names and coordinate types from header rows
    bodyparts_row = lines[1].strip().split(',')[1:]  # scorer repeated
    coords_row = lines[2].strip().split(',')[1:]     # bodypart names
    # lines[3] is x/y/likelihood labels

    # Parse the data
    table = []
    times = []
    for l in lines[4:]:
        values = l.strip().split(',')
        time = eval(values.pop(0))
        times.append(time)
        values = [eval(v) if v != '' else np.nan for v in values]
        table.append(values)

    table = np.array(table)
    times = np.array(times)

    # Build MultiIndex DataFrame: (bodypart, coord)
    data_dict = {}
    for i, (bp, coord) in enumerate(zip(coords_row, lines[3].strip().split(',')[1:])):
        data_dict[(bp, coord)] = table[:, i]

    df = pd.DataFrame(data_dict)
    df.columns = pd.MultiIndex.from_tuples(df.columns)

    return times, df

## Function to clean the data

In [ ]:
def remove_outliers_and_interpolate(df, framerate=30, interpolate=False):
    """Remove points > 2 std from mean, optionally interpolate gaps."""
    bodyparts = df.columns.get_level_values(0).unique().tolist()

    # Mask outliers (> 2 std from mean)
    df = df.mask(df.sub(df.mean()).div(df.std()).abs().gt(2))

    if interpolate:
        for bp in bodyparts:
            df[(bp, 'x')].interpolate(
                method='linear', limit_area='inside',
                limit=int(0.5 * framerate), inplace=True
            )
            df[(bp, 'y')].interpolate(
                method='linear', limit_area='inside',
                limit=int(0.5 * framerate), inplace=True
            )

    return df

## Function to filter by DLC likelihood

In [ ]:
def filter_by_likelihood(df, threshold=0.6):
    """Set x/y to NaN where DLC likelihood is below threshold."""
    bodyparts = df.columns.get_level_values(0).unique().tolist()
    df = df.copy()
    for bp in bodyparts:
        low_conf = df[(bp, 'likelihood')] < threshold
        df.loc[low_conf, (bp, 'x')] = np.nan
        df.loc[low_conf, (bp, 'y')] = np.nan
    return df

## Function to plot body part trajectories

In [ ]:
def plot_relevant_bps(df, times, fig_name, framerate=30):
    """Plot x and y trajectories for key body parts."""
    sel_bps = ['bodycentre', 'tailbase', 'lefthindlimb', 'righthindlimb']
    t_length = min(len(times), len(df))
    time_in_sec = t_length / framerate
    time = np.linspace(0, time_in_sec, t_length)

    fig, axs = plt.subplots(2, 2, figsize=(15, 6))
    for i, ax in enumerate(axs.flat):
        bp = sel_bps[i]
        if (bp, 'x') in df.columns:
            ax.plot(time, df[(bp, 'x')].iloc[:t_length], label='x')
            ax.plot(time, df[(bp, 'y')].iloc[:t_length], label='y')
        ax.set_title(bp)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.set_xlim(0, time_in_sec)
        ax.set_xlabel('Time (s)')
        ax.legend()
    fig.tight_layout()
    fig.savefig(fig_name, format='pdf', bbox_inches='tight')
    plt.show()

# Discover CSV files

Automatically find all DLC CSV files and extract the video/mouse name from the filename.

In [ ]:
csv_files = sorted(Path(data_dir).glob('*' + dlc_suffix + '.csv'))

# Extract video name by stripping the DLC suffix
# e.g. '648LDLC_Resnet50_...' -> '648L'
video_names = [f.stem.replace(dlc_suffix, '') for f in csv_files]

print(f'Found {len(csv_files)} CSV files:')
for name, path in zip(video_names, csv_files):
    print(f'  {name} <- {path.name}')

# Process each file

For each CSV: read, filter low-confidence points, remove outliers, interpolate, and save.

In [ ]:
all_results = {}

for video_name, csv_path in zip(video_names, csv_files):
    print(f'Processing {video_name} ...')

    # Read the DLC CSV
    times, df = build_dataframe(csv_path)
    print(f'  {len(df)} frames, {len(df.columns.get_level_values(0).unique())} body parts')

    # Filter out low-confidence detections
    df = filter_by_likelihood(df, threshold=0.6)

    # Remove outliers and interpolate small gaps
    df_clean = remove_outliers_and_interpolate(df, framerate=framerate, interpolate=True)

    # Store results
    all_results[video_name] = {
        'df': df_clean,
        'times': times,
    }

    # Plot trajectories
    fig_name = plots_dir + video_name + '_trajectories.pdf'
    plot_relevant_bps(df_clean, times, fig_name, framerate=framerate)

    # Save individual pickle
    individual_pickle = pickle_dir + video_name + '.pickle'
    with open(individual_pickle, 'wb') as handle:
        pickle.dump({'df': df_clean, 'times': times}, handle,
                    protocol=pickle.HIGHEST_PROTOCOL)
    print(f'  Saved -> {individual_pickle}')

print(f'\nDone! Processed {len(all_results)} files.')

# Save combined results

In [ ]:
combined_pickle = pickle_dir + 'all_results.pickle'
with open(combined_pickle, 'wb') as handle:
    pickle.dump(all_results, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved combined results -> {combined_pickle}')
print(f'Keys: {list(all_results.keys())}')

# Quick inspection

Check the data for a specific mouse to verify cleaning worked.

In [ ]:
# Pick the first mouse to inspect
inspect_name = video_names[0]
df_inspect = all_results[inspect_name]['df']

print(f'Inspecting: {inspect_name}')
print(f'Shape: {df_inspect.shape}')
print(f'Body parts: {df_inspect.columns.get_level_values(0).unique().tolist()}')
print(f'\nNaN percentage per body part:')
for bp in df_inspect.columns.get_level_values(0).unique():
    nan_pct = df_inspect[(bp, 'x')].isna().mean() * 100
    print(f'  {bp}: {nan_pct:.1f}%')